In [1]:
import os
import sys
import glob
import shutil
import argparse
import pandas as pd
import tqdm

In [10]:
annotation_fp = "/home/navi/sdb/parallel_kofam_scan_test_outs/wd/parallel_kofamscan/1000/30350551ec0c37f0544002c7053032c3_99d09fd0273fc0e27d54fe27304342f6/exec_annotation/part_009/annotations.tsv"
header_map_fp = "/home/navi/sdb/parallel_kofam_scan_test_outs/wd/parallel_kofamscan/1000/30350551ec0c37f0544002c7053032c3_99d09fd0273fc0e27d54fe27304342f6/validate_input/SM1510S13/header_map.parquet"

In [13]:
annotation_df = pd.read_csv(annotation_fp, sep = "\t", header = None, index_col = None, comment = "#")
annotation_df.columns = ["kofam_scan_annosgene","ORF","KO","thrshld","score","E_value","KO_definition"]
header_map_df = pd.read_parquet(header_map_fp)

In [14]:
annotation_df = pd.merge(annotation_df, header_map_df, left_on = "ORF", right_on = "uuid", how = "left")
del header_map_df
annotation_df = annotation_df[["id","KO","thrshld","score","E_value","KO_definition","kofam_scan_annosgene"]]

In [15]:
# filt res and apply top-hit here
min_e_val = 0.001
min_score = 100
# yhz: apply cut-off
annotation_df = annotation_df.query(f'(E_value <= {min_e_val}) and (score >= {min_score})')
# yhz: for each orf, tophit base on e-value and score
# id is the groupby key, so it is not in the columns to be selected. We use reset_index() to get it back.
annotation_df = annotation_df.sort_values(["E_value", 'score'], ascending = [True, False]).groupby('id')[["KO","thrshld","score","E_value"]].first().reset_index()

In [16]:
annotation_df

,id,KO,thrshld,score,E_value
0,SM1510S13_Contig_100008_1 # 555 # 1067 # -1 # ...,K03580,286.07,122.0,2.500000e-35
1,SM1510S13_Contig_1000856_1 # 102 # 359 # 1 # I...,K03625,88.60,121.9,3.200000e-35
2,SM1510S13_Contig_100171_2 # 478 # 1173 # -1 # ...,K03710,178.10,118.3,5.500000e-34
3,SM1510S13_Contig_100291_1 # 339 # 1091 # -1 # ...,K01809,33.17,177.4,5.900000e-52
4,SM1510S13_Contig_1002_5 # 4396 # 6090 # -1 # I...,K15363,183.23,307.3,2.400000e-91
...,...,...,...,...,...
3953,SM1510S13_Contig_9977_1 # 799 # 3093 # 1 # ID=...,K09461,658.73,1251.2,0.000000e+00
3954,SM1510S13_Contig_9983_1 # 135 # 3041 # 1 # ID=...,K21573,610.73,533.1,1.500000e-159
3955,SM1510S13_Contig_9994_3 # 2210 # 2830 # -1 # I...,K00969,121.70,205.3,1.600000e-60
3956,SM1510S13_Contig_999756_1 # 66 # 314 # 1 # ID=...,K23265,206.17,118.0,4.100000e-34
